# 🏭 PO Extraction Pipeline — Qwen2.5-14B + PyMuPDF + Surya OCR

**Purpose**: Extract structured JSON from multilingual Purchase Order PDFs (EN/DE/FR/JP/ZH)  
**GPU**: Requires T4 16GB (Kaggle free tier) or better  

## Instructions:
1. **Upload your PO PDFs** to Kaggle as a Dataset, or upload directly in the notebook
2. **Enable GPU**: Settings → Accelerator → GPU T4 x2
3. **Run all cells** — the output will be `results_kaggle.jsonl`
4. **Download** `results_kaggle.jsonl` and run local mapping on your machine

## Cell 1: Install Dependencies

In [ ]:
!pip install -q torch transformers accelerate bitsandbytes PyMuPDF json-repair
# Surya OCR is optional — only needed for scanned PDFs
# !pip install -q surya-ocr
print('✅ Dependencies installed')

## Cell 2: Upload PO Files
Upload your PDF/TXT files. You can:
- Add them as a Kaggle Dataset under `/kaggle/input/`
- Or upload via the file browser

In [ ]:
import os, glob

# === CONFIGURE THIS PATH ===
# Option A: Kaggle Dataset (recommended)
INPUT_FOLDER = '/kaggle/input/po-files/'    # Change to your dataset path

# Option B: Manual upload — uncomment and use the upload widget
# from google.colab import files
# uploaded = files.upload()
# INPUT_FOLDER = '/content/po_uploads/'
# os.makedirs(INPUT_FOLDER, exist_ok=True)
# for name, data in uploaded.items():
#     with open(os.path.join(INPUT_FOLDER, name), 'wb') as f: f.write(data)

OUTPUT_FILE = '/kaggle/working/results_kaggle.jsonl'

# List files
all_files = glob.glob(os.path.join(INPUT_FOLDER, '*'))
po_files = [f for f in all_files if f.lower().endswith(('.pdf', '.txt', '.png', '.jpg'))]
print(f'Found {len(po_files)} PO files in {INPUT_FOLDER}')
for f in po_files[:10]:
    print(f'  - {os.path.basename(f)} ({os.path.getsize(f)//1024} KB)')
if len(po_files) > 10:
    print(f'  ... and {len(po_files)-10} more')

## Cell 3: Load Qwen2.5-14B Model (4-bit Quantized)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

MODEL_ID = 'Qwen/Qwen2.5-14B-Instruct'
FALLBACK_MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4'
)

model_id = MODEL_ID
try:
    print(f'Loading {model_id}...')
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, device_map='auto', quantization_config=quantization_config
    )
    llm_pipe = pipeline('text-generation', model=model, tokenizer=tokenizer,
                        max_new_tokens=4096, do_sample=False, return_full_text=False)
    print(f'✅ {model_id} loaded on GPU')
except Exception as e:
    print(f'⚠️ 14B failed: {e}. Loading 7B fallback...')
    model_id = FALLBACK_MODEL_ID
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, device_map='auto', quantization_config=quantization_config
    )
    llm_pipe = pipeline('text-generation', model=model, tokenizer=tokenizer,
                        max_new_tokens=4096, do_sample=False, return_full_text=False)
    print(f'✅ Fallback {model_id} loaded on GPU')

# Quick GPU check
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM Used: {torch.cuda.memory_allocated(0)/1024**3:.1f} GB / {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

## Cell 4: Define Extraction Functions

In [ ]:
import json, re
from pathlib import Path
import fitz  # PyMuPDF

# ── Prompt ──
PROMPT = """You are a specialized Purchase Order extraction system. Extract structured data from the document below into JSON.

**RULES**:
- Output ONLY a single raw JSON object. No markdown, no explanation.
- Dates: DD/MM/YYYY format.
- If a value is not found, use null. NEVER output \"不明\", \"未提供\", \"unknown\", or \"N/A\".
- quantity must be a plain number (no units). Convert European format: 1.500 = 1500, 24.000 = 24000.
- material_code is an alphanumeric identifier, NEVER a weight like \"25Kg\" or price like \"70620 EUR\".

**LANGUAGE FIELD MAPPING**:
| Field | EN | DE | FR | JP | ZH |
|-------|----|----|----|----|-----|
| PO Number | PO No / Order No | Bestellnummer / Nr. | N° de commande | 注文番号 / 発注番号 | 订单号 / 采购订单号 |
| Order Date | Date | Bestelldatum | Date commande | 発注日 / 注文日 | 订单日期 |
| Delivery Date | Delivery Date | Lieferdatum / Liefertermin | Date de livraison | 納期 / 希望納期 | 交货日期 |
| Customer | Customer / Buyer | Kunde / Auftraggeber | Client | 買主 / 購入者 | 客户 / 买方 |
| Material Code | Part No / Item No | Artikelnr. / Material-Nr. | Réf. article | 品番 / 部品番号 | 物料编号 / 料号 |
| Material Desc | Description | Artikelbezeichnung / Bezeichnung | Désignation | 品名 / 品目 | 物料描述 / 品名 |
| Quantity | Qty | Menge | Quantité | 数量 | 数量 |
| Unit | UoM | Einheit / ME | Unité | 単位 | 单位 |

**CONTEXT**: Envalior B.V. (or DSM / Envalior) is ALWAYS the vendor/supplier. The OTHER company is the customer/buyer.

**JSON SCHEMA**:
{{\"header_fields\": {{\"po_number\": \"...\", \"order_date\": \"DD/MM/YYYY\", \"requested_delivery_date\": \"DD/MM/YYYY\", \"customer_id_or_name\": \"...\", \"vendor_name\": \"...\", \"ship_to_address\": \"...\", \"sold_to_address\": \"...\"}}, \"line_items\": [{{\"material_description\": \"...\", \"quantity\": \"...\", \"unit\": \"...\", \"delivery_date\": \"DD/MM/YYYY\", \"material_code\": \"...\"}}]}}

**DOCUMENT TEXT**:
{content}

JSON:"""


# ── Text Extraction ──
def extract_text(file_path):
    path = Path(file_path)
    if path.suffix.lower() == '.txt':
        return path.read_text(encoding='utf-8', errors='replace').strip(), 'direct_text'
    if path.suffix.lower() == '.pdf':
        doc = fitz.open(str(path))
        texts = [page.get_text('text') for page in doc]
        doc.close()
        full = '\n\n'.join(t for t in texts if t.strip())
        if len(full) > 100:
            return full, 'pymupdf_native'
        return full, 'pymupdf_sparse'
    return '', 'unsupported'


# ── JSON Repair ──
def parse_json_robust(raw):
    if not raw: return {'error': 'empty'}
    cleaned = re.sub(r'```json\\s*', '', raw.strip(), flags=re.IGNORECASE)
    cleaned = re.sub(r'```\\s*', '', cleaned)
    start = cleaned.find('{')
    if start == -1: return {'error': 'no JSON', 'raw': raw[:300]}
    bal, end = 0, -1
    for i in range(start, len(cleaned)):
        if cleaned[i] == '{': bal += 1
        elif cleaned[i] == '}': bal -= 1
        if bal == 0: end = i; break
    snippet = cleaned[start:end+1] if end != -1 else cleaned[start:]
    try:
        obj = json.loads(snippet)
        return obj.get('data', obj) if 'data' in obj and isinstance(obj.get('data'), dict) else obj
    except json.JSONDecodeError: pass
    try:
        from json_repair import repair_json
        repaired = repair_json(snippet, return_objects=True)
        if isinstance(repaired, dict):
            return repaired.get('data', repaired) if 'data' in repaired else repaired
    except: pass
    try:
        fixed = re.sub(r',\\s*}', '}', snippet)
        fixed = re.sub(r',\\s*]', ']', fixed).replace(\"'\", '\"')
        return json.loads(fixed)
    except: return {'error': 'JSON parse failed', 'raw': snippet[:300]}


# ── Post-Processing ──
UNKNOWN_MARKERS = {'不明', '未提供', 'unknown', 'N/A', 'n/a', 'なし', '无', '未知'}

def clean_unknowns(obj):
    if isinstance(obj, dict): return {k: clean_unknowns(v) for k, v in obj.items()}
    if isinstance(obj, list): return [clean_unknowns(i) for i in obj]
    if isinstance(obj, str) and obj.strip() in UNKNOWN_MARKERS: return None
    return obj

print('✅ Functions defined')

## Cell 5: 🚀 Run Extraction

In [ ]:
import time

po_files_sorted = sorted(po_files)
results = []
success_count = 0
error_count = 0

print(f'Processing {len(po_files_sorted)} files...\n')
start_time = time.time()

with open(OUTPUT_FILE, 'w', encoding='utf-8') as out_f:
    for idx, fpath in enumerate(po_files_sorted, 1):
        fname = os.path.basename(fpath)
        print(f'\n── [{idx}/{len(po_files_sorted)}] {fname} ──')

        # Skip tiny images (logos)
        if fname.lower().endswith(('.png', '.jpg', '.jpeg')) and os.path.getsize(fpath) < 80000:
            print(f'  ⏭️ Skipped (too small, likely logo)')
            continue

        # Extract text
        text, engine = extract_text(fpath)
        if len(text.strip()) < 10:
            print(f'  ⚠️ No text extracted')
            rec = {'source_file': fname, 'error': 'Empty text', 'engine': engine}
            out_f.write(json.dumps(rec, ensure_ascii=False) + '\n')
            error_count += 1
            continue
        print(f'  [Text] {len(text)} chars via {engine}')

        # LLM extraction
        content = text[:8000]  # Truncate for context window
        prompt = PROMPT.format(content=content)
        try:
            out = llm_pipe(prompt)
            raw_output = out[0]['generated_text']
            extracted = parse_json_robust(raw_output)
        except Exception as e:
            print(f'  ❌ LLM error: {e}')
            rec = {'source_file': fname, 'error': str(e), 'engine': engine}
            out_f.write(json.dumps(rec, ensure_ascii=False) + '\n')
            error_count += 1
            continue

        # Post-process
        extracted = clean_unknowns(extracted)
        extracted['source_file'] = fname
        extracted['ocr_engine'] = engine

        header = extracted.get('header_fields', {})
        items = extracted.get('line_items', [])
        po = header.get('po_number', '?')
        cust = header.get('customer_id_or_name', '?')
        print(f'  ✅ PO: {po} | Customer: {cust} | Items: {len(items)}')

        out_f.write(json.dumps(extracted, ensure_ascii=False) + '\n')
        success_count += 1

elapsed = time.time() - start_time
print(f'\n{"="*60}')
print(f'✅ DONE — {success_count} success, {error_count} errors in {elapsed:.0f}s')
print(f'Output: {OUTPUT_FILE}')
print(f'{"="*60}')

## Cell 6: Preview Results

In [ ]:
import pandas as pd

records = []
with open(OUTPUT_FILE, 'r') as f:
    for line in f:
        if line.strip():
            obj = json.loads(line)
            header = obj.get('header_fields', {})
            items = obj.get('line_items', [])
            records.append({
                'source_file': obj.get('source_file', ''),
                'po_number': header.get('po_number', ''),
                'customer': header.get('customer_id_or_name', ''),
                'vendor': header.get('vendor_name', ''),
                'order_date': header.get('order_date', ''),
                'delivery_date': header.get('requested_delivery_date', ''),
                'num_items': len(items),
                'engine': obj.get('ocr_engine', ''),
                'error': obj.get('error', ''),
            })

df = pd.DataFrame(records)
print(f'Total records: {len(df)}')
print(f'Successful: {len(df[df.error == ""])}')
print(f'Errors: {len(df[df.error != ""])}')
print()
df

## Cell 7: Download Results
Download the JSONL file. Then on your local machine, run:
```
python reenrich_results.py --input results_kaggle.jsonl --output results_kaggle_enriched.jsonl
python run_test_export.py
```

In [ ]:
# For Google Colab:
# from google.colab import files
# files.download(OUTPUT_FILE)

# For Kaggle: the file is in /kaggle/working/ — use the Output tab to download
print(f'\n📥 Download your results from: {OUTPUT_FILE}')
print('On Kaggle: Go to the Output tab on the right side panel')
print('On Colab: Uncomment the files.download() line above')